> **🎯 Important**
>
**Durée estimée** : 10 à 12 heures  
**Prérequis** : toutes les notions du Module 6 (6.1 à 6.6)  
**Objectif** : construire un classifieur d'images professionnel combinant CNN from scratch, transfer learning, data augmentation et gestion de classes déséquilibrées


> **💡 Astuce**
>
## 💻 Ressources
- **GPU recommandé** : l'entraînement sera 20-50× plus rapide. Utilise **Google Colab** (GPU T4 gratuit).
- **Sans GPU** : c'est faisable, mais prévois des temps d'entraînement plus longs (~30 min par modèle).


# 🎯 Contexte

Tu es **data scientist** dans une startup, **MetalCheck**, qui développe une solution de **contrôle qualité automatisé** pour l'industrie métallurgique. 

> *« On a des opérateurs qui passent leur journée à inspecter des pièces à l'œil. C'est fatiguant, répétitif, et on rate des défauts. On veut un modèle qui prenne une photo et dise : **OK**, **rayure**, ou **fissure** (critique). Voilà les 520 photos qu'on a pu étiqueter — vas-y ! »*

## 💼 Les objectifs business

1. **Classifier automatiquement** les pièces en 3 catégories : `ok`, `rayure`, `fissure`
2. **Haute précision** sur les fissures (critique pour la sécurité)
3. **Modèle industrialisable** : rapide, robuste, reproductible
4. **Rapport clair** pour l'équipe production

## 📊 Le dataset

520 images au total, déjà split en train/test :

| Classe | Train | Test | Total | Description |
|---|---:|---:|---:|---|
| `ok` | 320 | 80 | 400 | Pièce saine |
| `rayure` | 64 | 16 | 80 | Défaut mineur |
| `fissure` | 32 | 8 | 40 | Défaut critique |
| **Total** | **416** | **104** | **520** | |

> **⚠️ Attention**
>
## ⚠️ Particularités importantes
- Dataset **fortement déséquilibré** : 77% de `ok`, 15% de `rayure`, 8% de `fissure`
- **Peu de fissures** : risque de les manquer si on ne fait rien
- Images **224×224 RGB** : parfait pour transfer learning
- Split train/test **déjà fait** (pas touche au test set avant la fin !)


## 🗺️ Plan du TP

Le TP se structure en **7 parties** qui suivent le workflow DL professionnel :

1. **Exploration** — comprendre les données
2. **Pipeline de données** — DataLoaders + data augmentation
3. **Baseline CNN from scratch** — voir où on part
4. **Transfer learning** — feature extraction
5. **Fine-tuning** — améliorer encore
6. **Gestion du déséquilibre** — améliorer les prédictions sur fissure
7. **Évaluation finale et rapport**

# 🛠️ Mise en route

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torchvision.datasets import ImageFolder

from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ PyTorch {torch.__version__}, device : {device}")

In [ ]:
DATA_DIR = Path("ressources_tp/pieces")

In [ ]:
DATA_DIR = Path("ressources_tp/pieces")

---

# Partie 1 — Exploration

## ✏️ Étape 1.1 — Structure du dataset

> **ℹ️ Note**
>
## 📝 À faire

1. Affiche l'**arborescence** du dataset (`train/ok/`, `train/rayure/`, etc.)
2. Compte le **nombre d'images** par classe et par split
3. Affiche le déséquilibre en pourcentages

In [ ]:
# TODO: Étape 1.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 1.2 — Visualiser quelques images

> **ℹ️ Note**
>
## 📝 À faire

1. Affiche une **grille 3×4** avec 4 images de chaque classe
2. Observations : qu'est-ce qui distingue visuellement les classes ?

In [ ]:
# TODO: Étape 1.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 2 — Pipeline de données

## ✏️ Étape 2.1 — Transformations

> **ℹ️ Note**
>
## 📝 À faire

Crée **deux pipelines** de transformations :

1. **`transform_train`** : avec data augmentation
   - Resize 224
   - RandomHorizontalFlip (OK pour les pièces, symétriques)
   - RandomRotation ±15°
   - ColorJitter légère (brightness, contrast)
   - ToTensor + Normalize avec stats ImageNet

2. **`transform_test`** : sans augmentation
   - Resize 224 + ToTensor + Normalize

Pourquoi différents pipelines ? Parce que la data augmentation ne doit **pas** s'appliquer au test (on veut évaluer sur des images "telles quelles").

In [ ]:
# TODO: Étape 2.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 2.2 — DataLoaders

> **ℹ️ Note**
>
## 📝 À faire

1. Charge les datasets avec **`ImageFolder`** (très pratique : il découvre les classes depuis les dossiers)
2. Crée 2 DataLoaders : `train_loader` (batch=32, shuffle=True) et `test_loader` (batch=32)
3. Affiche les **classes découvertes** et leur **mapping** (classe → index)
4. Affiche la shape d'un batch

In [ ]:
# TODO: Étape 2.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 3 — Baseline : CNN from scratch

Avant de sortir l'artillerie lourde (transfer learning), essayons **un CNN from scratch** pour voir où on part.

## ✏️ Étape 3.1 — Définir et entraîner un petit CNN

> **ℹ️ Note**
>
## 📝 À faire

Définis un **CNN simple** :
- 3 blocs `Conv → BN → ReLU → MaxPool`
- Canaux : 32 → 64 → 128
- Puis flatten + FC(256) + FC(3)
- Dropout 0.3 dans la tête

Entraîne **10 époques** avec Adam (lr=1e-3). Affiche l'accuracy test à chaque époque.

In [ ]:
# TODO: Étape 3.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 3.2 — Analyser les erreurs (matrice de confusion)

> **ℹ️ Note**
>
## 📝 À faire

1. Calcule la **matrice de confusion** sur le test set
2. Affiche-la en heatmap
3. **Question critique** : le modèle rate-t-il des `fissure` ?

In [ ]:
# TODO: Étape 3.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 4 — Transfer learning (feature extraction)

Passons à la **technique pro** : un modèle pré-entraîné sur ImageNet.

## ✏️ Étape 4.1 — ResNet-18 avec feature extraction

> **ℹ️ Note**
>
## 📝 À faire

1. Charge `resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)`
2. **Gèle tous les paramètres**
3. Remplace `fc` par un `Linear` vers 3 classes
4. Vérifie combien de paramètres sont **entraînables** (devrait être ~1500 = 512×3 + 3)
5. Entraîne 10 époques
6. Compare à la baseline from scratch

In [ ]:
# TODO: Étape 4.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 4.2 — Matrice de confusion du modèle TL

> **ℹ️ Note**
>
## 📝 À faire

Refais la matrice de confusion avec le modèle transfer learning. Est-ce mieux sur `fissure` ?

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 5 — Fine-tuning

Feature extraction = gel total. **Fine-tuning** = on dégèle les dernières couches pour **adapter** au domaine.

## ✏️ Étape 5.1 — Fine-tuning de layer4

> **ℹ️ Note**
>
## 📝 À faire

1. Recharge un ResNet-18 pré-entraîné
2. Gèle tout SAUF **`layer4`** et **`fc`**
3. Utilise un optimizer avec **2 learning rates** :
   - `layer4` : lr = 1e-4 (petit, pour affiner)
   - `fc` : lr = 1e-3 (normal, pour la nouvelle tête)
4. Entraîne 10 époques
5. Compare à feature extraction

In [ ]:
# TODO: Étape 5.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 6 — Gérer le déséquilibre des classes

Problème : **8% de fissures** seulement. Le modèle peut être "tenté" de négliger cette classe.

## 🎯 Deux techniques

1. **Weighted loss** : pondérer plus fort les erreurs sur les classes rares
2. **WeightedRandomSampler** : dans le DataLoader, piocher plus souvent les classes rares

## ✏️ Étape 6.1 — Weighted loss

> **ℹ️ Note**
>
## 📝 À faire

1. Calcule les **poids** pour chaque classe : inversement proportionnels à leur fréquence
2. Refais l'entraînement **feature extraction** avec `nn.CrossEntropyLoss(weight=poids)`
3. Compare le **recall sur `fissure`** avant/après

In [ ]:
# TODO: Étape 6.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 7 — Évaluation finale et rapport

## ✏️ Étape 7.1 — Choisir et évaluer le modèle final

> **ℹ️ Note**
>
## 📝 À faire

1. Choisis ton **meilleur modèle** (probablement fine-tuning + weighted loss)
2. Calcule **toutes les métriques** détaillées
3. Trace la **matrice de confusion** finale
4. Affiche **les 6 erreurs les plus "sûres"** (le modèle se trompait avec haute confiance)

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 7.2 — Sauvegarder le modèle

> **ℹ️ Note**
>
## 📝 À faire

Sauvegarde le modèle final + la config (classes, stats de normalisation) pour qu'il puisse être **rechargé en production**.

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

# 🎓 Bilan du TP

## 🏆 Ce que tu viens d'accomplir

Tu as mené un **projet Deep Learning complet** :

- ✅ Exploration et compréhension des données
- ✅ Pipeline de données avec data augmentation
- ✅ Baseline CNN from scratch (pour comparer)
- ✅ Transfer learning (feature extraction)
- ✅ Fine-tuning (ajustement plus fin)
- ✅ Gestion du déséquilibre (weighted loss)
- ✅ Évaluation détaillée avec matrice de confusion
- ✅ Sauvegarde pour production

C'est **exactement** le workflow d'un projet vision en entreprise.

## 📊 Résumé des performances

Tableau à remplir avec **tes** résultats :

| Approche | Accuracy | Recall fissure | Temps train |
|---|:---:|:---:|:---:|
| CNN from scratch | ~82% | ~70% | ~1 min |
| Transfer learning (FE) | ~95% | ~85% | ~1 min |
| Fine-tuning | ~97% | ~90% | ~2 min |
| + Weighted loss | ~96% | ~95% | ~1 min |

## 💡 Pistes pour aller plus loin

1. **Modèles plus gros** : ResNet-50, EfficientNet-B3, ViT-B/16
2. **Augmentation plus agressive** : CutMix, MixUp, AutoAugment
3. **Ensemble de modèles** : moyenne des prédictions de 3 modèles différents
4. **Test Time Augmentation** : prédire sur 5 versions augmentées et moyenner
5. **Learning rate scheduling** + **Early stopping**
6. **Explicabilité** : Grad-CAM pour visualiser où le modèle regarde

## 🎯 Ce qu'il faut retenir

> **En vision par ordinateur, le transfer learning est incontournable.**
> 
> Avec 400 images et 1 heure de code, on atteint des performances **inaccessibles** à un modèle from scratch en 10 fois plus de données.

## 🚀 La suite du parcours

Tu as complété le **Module 6 — Deep Learning** ! 🎉

Les prochains modules abordent :

- **Module 7** — LLM et IA générative (embeddings, RAG, agents)
- **Module 8** — MLOps (déploiement, monitoring, MLflow)

Tu as maintenant **toutes les briques** d'un data scientist moderne :
- ✅ Fondations (M1-3)
- ✅ ML supervisé + non supervisé (M4-5)
- ✅ Deep Learning (M6) ← **Tu es ici**
- ⏳ LLM + RAG (M7)
- ⏳ MLOps (M8)

Bravo pour ce TP exigeant, tu peux être fier·ère de toi ! 👏